# Template-Aware vs. Random Dataset Splitting

This notebook demonstrates the structural data leakage that occurs when using a naive random split on the Juliet dataset, compared to our proposed template-aware split.

## 1. Load Data
We load the vulnerability dataset to analyze the distribution of code templates.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load a sample of the Juliet dataset
df = pd.read_csv("../data/juliet_cwe_sample(1%).csv")

# Assuming the dataset has a column like "template_id" or we can infer it from "test_case_id" or file name.
# For demonstration purposes of this structural evaluation:
print(df.head())

## 2. Simulate Random Split Leakage
If we randomly split the data 80:20, we see how many test samples share the exact same template structure as samples in the training set.

In [ ]:
from sklearn.model_selection import train_test_split

if "template" in df.columns:
    # Random split
    train_rand, test_rand = train_test_split(df, test_size=0.2, random_state=42)
    
    # Calculate Leakage: Test templates that exist in train set
    train_templates = set(train_rand["template"])
    test_templates = set(test_rand["template"])
    leaked_templates = test_templates.intersection(train_templates)
    leakage_percent_random = (len(leaked_templates) / len(test_templates)) * 100
    print(f"Random Split Data Leakage: {leakage_percent_random:.2f}% of test templates were seen during training!")

    # Template-Aware Split
    unique_templates = df["template"].unique()
    train_templates, test_templates = train_test_split(unique_templates, test_size=0.2, random_state=42)
    
    leakage_percent_template = len(set(test_templates).intersection(set(train_templates))) * 100
    print(f"Template-Aware Split Data Leakage: {leakage_percent_template:.2f}% (Perfect Isolation)")

    # Plotting
    plt.figure(figsize=(8, 5))
    sns.barplot(x=["Random Split", "Template-Aware Split"], y=[leakage_percent_random, leakage_percent_template], palette=["red", "green"])
    plt.ylabel("Structural Data Leakage (%)")
    plt.title("Data Leakage Comparison: Random vs Template-Aware")
    plt.savefig("../docs/random_vs_template_leakage.png")
    plt.show()
else:
    print("Template column not found, please adjust column mapping.")